In [39]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/optimal-fertilizers-prediction/__results__.html
/kaggle/input/optimal-fertilizers-prediction/__notebook__.ipynb
/kaggle/input/optimal-fertilizers-prediction/__output__.json
/kaggle/input/optimal-fertilizers-prediction/submission_upgraded_domain.csv
/kaggle/input/optimal-fertilizers-prediction/custom.css
/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv
/kaggle/input/playground-series-s5e6/sample_submission.csv
/kaggle/input/playground-series-s5e6/train.csv
/kaggle/input/playground-series-s5e6/test.csv


In [40]:
import numpy as np
import pandas as pd
import gc
import warnings
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier

In [41]:
warnings.simplefilter(action='ignore')
print("Loading all datasets...")

train = pd.read_csv("/kaggle/input/playground-series-s5e6/train.csv")
test = pd.read_csv("/kaggle/input/playground-series-s5e6/test.csv")
original = pd.read_csv("/kaggle/input/fertilizer-prediction/Fertilizer Prediction.csv")
submission = pd.read_csv("/kaggle/input/playground-series-s5e6/sample_submission.csv")


Loading all datasets...


In [42]:
original.head()

,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,32,51,41,Red,Ground Nuts,7,3,19,14-35-14
1,35,58,35,Black,Cotton,4,14,16,Urea
2,27,55,43,Sandy,Sugarcane,28,0,17,20-20
3,33,56,56,Loamy,Ground Nuts,37,5,24,28-28
4,32,70,60,Red,Ground Nuts,4,6,9,14-35-14


In [43]:
original.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   Temparature      100000 non-null  int64 
 1   Humidity         100000 non-null  int64 
 2   Moisture         100000 non-null  int64 
 3   Soil Type        100000 non-null  object
 4   Crop Type        100000 non-null  object
 5   Nitrogen         100000 non-null  int64 
 6   Potassium        100000 non-null  int64 
 7   Phosphorous      100000 non-null  int64 
 8   Fertilizer Name  100000 non-null  object
dtypes: int64(6), object(3)
memory usage: 6.9+ MB


In [44]:
train.head()

,id,Temparature,Humidity,Moisture,Soil Type,Crop Type,Nitrogen,Potassium,Phosphorous,Fertilizer Name
0,0,37,70,36,Clayey,Sugarcane,36,4,5,28-28
1,1,27,69,65,Sandy,Millets,30,6,18,28-28
2,2,29,63,32,Sandy,Millets,24,12,16,17-17-17
3,3,35,62,54,Sandy,Barley,39,12,4,10-26-26
4,4,35,58,43,Red,Paddy,37,2,16,DAP


In [45]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 750000 entries, 0 to 749999
Data columns (total 10 columns):
 #   Column           Non-Null Count   Dtype 
---  ------           --------------   ----- 
 0   id               750000 non-null  int64 
 1   Temparature      750000 non-null  int64 
 2   Humidity         750000 non-null  int64 
 3   Moisture         750000 non-null  int64 
 4   Soil Type        750000 non-null  object
 5   Crop Type        750000 non-null  object
 6   Nitrogen         750000 non-null  int64 
 7   Potassium        750000 non-null  int64 
 8   Phosphorous      750000 non-null  int64 
 9   Fertilizer Name  750000 non-null  object
dtypes: int64(7), object(3)
memory usage: 57.2+ MB


In [46]:
print("Preparing and augmenting data...")
for df in [train, test, original]:
    df.rename(columns={'Temparature': 'Temperature'}, inplace=True)

original_copy = original.copy()
for i in range(6):
    original = pd.concat([original, original_copy], axis=0, ignore_index=True)
print("Augmented external dataset size: {len(original)} rows.")

Preparing and augmenting data...
Augmented external dataset size: {len(original)} rows.


In [47]:
def create_hybrid_features(df):
    # This function creates BOTH numerical and categorical features
    
    # Part 1: Our "Champion" Numerical Features
    epsilon = 1e-6
    df['N_P_Ratio'] = df['Nitrogen'] / (df['Phosphorous'] + epsilon)
    df['P_K_Ratio'] = df['Phosphorous'] / (df['Potassium'] + epsilon)
    df['N_K_Ratio'] = df['Nitrogen'] / (df['Potassium'] + epsilon)
    df['Total_Nutrients'] = df['Nitrogen'] + df['Phosphorous'] + df['Potassium']
    df['Temp_Humidity_Index'] = df['Temperature'] * df['Humidity']
    df['Soil_Quality_Index'] = df['Moisture'] / (df['Temperature'] + epsilon)
    
    # Part 2: The Winning Script's Categorical Binning Features
    numerical_cols_original = ['Temperature', 'Humidity', 'Moisture', 'Nitrogen', 'Phosphorous', 'Potassium']
    for col in numerical_cols_original:
        df[f'{col}_cat_bin'] = df[col].astype(str) # Create new binned columns
        
    return df

print("Applying HYBRID feature engineering...")
train = create_hybrid_features(train)
test = create_hybrid_features(test)
original = create_hybrid_features(original)

Applying HYBRID feature engineering...


In [48]:
print("Encoding categorical features...")
# Identify only the non-numeric features for encoding
categorical_cols = [col for col in train.columns if train[col].dtype == 'object']
if "Fertilizer Name" in categorical_cols:
    categorical_cols.remove("Fertilizer Name")

for col in categorical_cols:
    combined_cats = pd.concat([train[col], test[col], original[col]]).unique()
    le = LabelEncoder().fit(combined_cats)
    train[col] = le.transform(train[col])
    test[col] = le.transform(test[col])
    original[col] = le.transform(original[col])
    
# Encode the target
target_encoder = LabelEncoder()
train["Fertilizer Name"] = target_encoder.fit_transform(train["Fertilizer Name"])
original["Fertilizer Name"] = target_encoder.transform(original["Fertilizer Name"])

# Set ONLY the encoded columns to 'category' dtype for XGBoost
for col in categorical_cols:
    train[col] = train[col].astype("category")
    test[col] = test[col].astype("category")
    original[col] = original[col].astype("category")

Encoding categorical features...


In [49]:
print("Preparing data for training...")
X = train.drop(columns=["id", "Fertilizer Name"])
y = train["Fertilizer Name"]
X_test = test.drop(columns=["id"])
X_original = original.drop(columns=["Fertilizer Name"])
y_original = original["Fertilizer Name"]

# Ensure columns are in the same order
X_test = X_test[X.columns]
X_original = X_original[X.columns]

# The same winning parameters
params = {
    'objective': 'multi:softprob', 'num_class': y.nunique(), 'max_depth': 7,
    'learning_rate': 0.03, 'subsample': 0.8, 'max_bin': 128, 'colsample_bytree': 0.3,
    'tree_method': 'hist', 'random_state': 42, 'eval_metric': 'mlogloss',
    'device': "cuda", 'enable_categorical': True, 'n_estimators': 10000
}

FOLDS = 5
skf = StratifiedKFold(n_splits=FOLDS, shuffle=True, random_state=42)
test_preds = np.zeros(shape=(len(test), y.nunique()))

print("\nStarting 5-Fold training with HYBRID features...")
for i, (train_idx, valid_idx) in enumerate(skf.split(X, y)):
    print('#' * 15, f" FOLD {i+1} ", '#' * 15)
    
    x_train, x_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    x_train_aug = pd.concat([x_train, X_original], axis=0)
    y_train_aug = pd.concat([y_train, y_original], axis=0)
    
    model = XGBClassifier(**params)
    model.fit(
        x_train_aug, y_train_aug,
        eval_set=[(x_valid, y_valid)],
        early_stopping_rounds=50,
        verbose=1000
    )
    
    test_preds += model.predict_proba(X_test) / FOLDS
    del x_train, x_valid, y_train, y_valid, x_train_aug, y_train_aug, model
    gc.collect()

Preparing data for training...

Starting 5-Fold training with HYBRID features...
###############  FOLD 1  ###############
[0]	validation_0-mlogloss:1.94571
[1000]	validation_0-mlogloss:1.89522
[2000]	validation_0-mlogloss:1.88621
[2296]	validation_0-mlogloss:1.88577
###############  FOLD 2  ###############
[0]	validation_0-mlogloss:1.94570
[1000]	validation_0-mlogloss:1.89516
[2000]	validation_0-mlogloss:1.88568
[2409]	validation_0-mlogloss:1.88511
###############  FOLD 3  ###############
[0]	validation_0-mlogloss:1.94571
[1000]	validation_0-mlogloss:1.89469
[2000]	validation_0-mlogloss:1.88515
[2444]	validation_0-mlogloss:1.88454
###############  FOLD 4  ###############
[0]	validation_0-mlogloss:1.94570
[1000]	validation_0-mlogloss:1.89598
[2000]	validation_0-mlogloss:1.88693
[2372]	validation_0-mlogloss:1.88625
###############  FOLD 5  ###############
[0]	validation_0-mlogloss:1.94572
[1000]	validation_0-mlogloss:1.89501
[2000]	validation_0-mlogloss:1.88588
[2446]	validation_0-mloglo

In [51]:
categorical_cols = X.select_dtypes(include=['category', 'object']).columns.tolist()


1. **Catboost**

In [52]:
!pip install optuna --quiet


In [53]:
# import optuna
# import pandas as pd
# import numpy as np
# from sklearn.model_selection import StratifiedKFold
# from catboost import CatBoostClassifier
# from sklearn.metrics import log_loss

# print("Preparing data for CatBoost training...")

# # Drop unnecessary columns and extract labels
# cb_data = train.drop(columns=["id", "Fertilizer Name"])
# cb_labels = train["Fertilizer Name"]
# cb_test = test.drop(columns=["id"])

# cb_extra_data = original.drop(columns=["Fertilizer Name"])
# cb_extra_labels = original["Fertilizer Name"]

# # Convert object columns to 'category'
# for df in [cb_data, cb_test, cb_extra_data]:
#     for col in df.columns:
#         if df[col].dtype == 'object':
#             df[col] = df[col].astype('category')

# # Get categorical column names (those actually of dtype 'category')
# cb_cat_features = cb_data.select_dtypes(include='category').columns.tolist()

# # Align columns
# cb_test = cb_test[cb_data.columns].copy()
# cb_extra_data = cb_extra_data[cb_data.columns].copy()

# def objective(trial):
#     params = {
#         'iterations': 10000,
#         'learning_rate': trial.suggest_float("learning_rate", 0.01, 0.1),
#         'depth': trial.suggest_int("depth", 4, 12),
#         'l2_leaf_reg': trial.suggest_float("l2_leaf_reg", 1, 10),
#         'bagging_temperature': trial.suggest_float("bagging_temperature", 0, 1),
#         'random_strength': trial.suggest_float("random_strength", 1e-9, 10.0, log=True),
#         'border_count': trial.suggest_int("border_count", 32, 255),
#         'loss_function': 'MultiClass',
#         'eval_metric': 'MultiClass',
#         'task_type': 'GPU',
#         'devices': '0',
#         'early_stopping_rounds': 50,
#         'verbose': False,
#         'random_seed': 42
#     }

#     skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
#     scores = []

#     for train_idx, valid_idx in skf.split(cb_data, cb_labels):
#         X_train = cb_data.iloc[train_idx].copy()
#         y_train = cb_labels.iloc[train_idx]
#         X_valid = cb_data.iloc[valid_idx].copy()
#         y_valid = cb_labels.iloc[valid_idx]

#         # Augment training data
#         X_train_aug = pd.concat([X_train, cb_extra_data], axis=0).reset_index(drop=True)
#         y_train_aug = pd.concat([y_train, cb_extra_labels], axis=0).reset_index(drop=True)

#         model = CatBoostClassifier(**params)
#         model.fit(
#             X_train_aug, y_train_aug,
#             eval_set=(X_valid, y_valid),
#             cat_features=cb_cat_features
#         )

#         preds = model.predict_proba(X_valid)
#         score = log_loss(y_valid, preds)
#         scores.append(score)

#     return np.mean(scores)




In [54]:
# # Run the optimization
# study = optuna.create_study(direction='minimize')
# study.optimize(objective, n_trials=100, show_progress_bar=True)

# # Show best result
# print("Best parameters:", study.best_params)
# print("Best log loss:", study.best_value)

Trial 18 finished with value: 1.9174488243878385 and parameters: {'learning_rate': 0.023611018074820644, 'depth': 6, 'l2_leaf_reg': 8.525013278321158, 'bagging_temperature': 0.42901588028226817, 'random_strength': 9.071021020849433e-07, 'border_count': 70}. Best is trial 18 with value: 1.9174488243878385.

In [55]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
import numpy as np
import pandas as pd
import gc

print("Preparing data for CatBoost training...")
cb_data = train.drop(columns=["id", "Fertilizer Name"])
cb_labels = train["Fertilizer Name"]
cb_test_data = test.drop(columns=["id"])
cb_extra_data = original.drop(columns=["Fertilizer Name"])
cb_extra_labels = original["Fertilizer Name"]

# Ensure columns are aligned
cb_test_data = cb_test_data[cb_data.columns]
cb_extra_data = cb_extra_data[cb_data.columns]

# Handle categorical features
for col in cb_data.columns:
    if cb_data[col].dtype == 'object':
        cb_data[col] = cb_data[col].astype('category')
        cb_test_data[col] = cb_test_data[col].astype('category')
        cb_extra_data[col] = cb_extra_data[col].astype('category')

cb_cat_features = cb_data.select_dtypes(include='category').columns.tolist()

# CatBoost GPU parameters
cb_params = {
    'learning_rate': 0.023611018074820644,
     'depth': 6,
     'l2_leaf_reg': 8.525013278321158,
     'bagging_temperature': 0.42901588028226817, 
     'random_strength': 9.071021020849433e-07,
     'border_count': 70,
    'iterations': 10000,
    
    
    'loss_function': 'MultiClass',
    'eval_metric': 'MultiClass',
    'task_type': 'GPU',
    'devices': '0',
    'random_seed': 42,
    'early_stopping_rounds': 50,
    'verbose': 1000
}

cb_folds = 5
cb_skf = StratifiedKFold(n_splits=cb_folds, shuffle=True, random_state=42)
cb_predictions = np.zeros((len(cb_test_data), cb_labels.nunique()))

print("\nStarting 5-Fold training with CatBoost using GPU...")
for fold, (cb_train_idx, cb_valid_idx) in enumerate(cb_skf.split(cb_data, cb_labels)):
    print('#' * 15, f" CATBOOST FOLD {fold+1} ", '#' * 15)
    
    cb_fold_train_data = cb_data.iloc[cb_train_idx]
    cb_fold_valid_data = cb_data.iloc[cb_valid_idx]
    cb_fold_train_labels = cb_labels.iloc[cb_train_idx]
    cb_fold_valid_labels = cb_labels.iloc[cb_valid_idx]

    cb_aug_train_data = pd.concat([cb_fold_train_data, cb_extra_data], axis=0)
    cb_aug_train_labels = pd.concat([cb_fold_train_labels, cb_extra_labels], axis=0)

    cb_model = CatBoostClassifier(**cb_params)

    cb_model.fit(
        cb_aug_train_data, cb_aug_train_labels,
        eval_set=(cb_fold_valid_data, cb_fold_valid_labels),
        cat_features=cb_cat_features
    )

    cb_predictions += cb_model.predict_proba(cb_test_data) / cb_folds

    del cb_fold_train_data, cb_fold_valid_data, cb_fold_train_labels, cb_fold_valid_labels
    del cb_aug_train_data, cb_aug_train_labels, cb_model
    gc.collect()


Preparing data for CatBoost training...

Starting 5-Fold training with CatBoost using GPU...
###############  CATBOOST FOLD 1  ###############
0:	learn: 1.9458212	test: 1.9458019	best: 1.9458019 (0)	total: 73.2ms	remaining: 12m 12s
1000:	learn: 1.9083938	test: 1.9277554	best: 1.9277554 (1000)	total: 54.6s	remaining: 8m 10s
2000:	learn: 1.8860821	test: 1.9232442	best: 1.9232442 (2000)	total: 1m 46s	remaining: 7m 6s
3000:	learn: 1.8689577	test: 1.9206717	best: 1.9206717 (3000)	total: 2m 38s	remaining: 6m 9s
4000:	learn: 1.8531435	test: 1.9189252	best: 1.9189252 (4000)	total: 3m 29s	remaining: 5m 14s
5000:	learn: 1.8381800	test: 1.9176921	best: 1.9176921 (5000)	total: 4m 20s	remaining: 4m 20s
6000:	learn: 1.8230481	test: 1.9167862	best: 1.9167842 (5997)	total: 5m 12s	remaining: 3m 28s
bestTest = 1.91616875
bestIteration = 6932
Shrink model to first 6933 iterations.
###############  CATBOOST FOLD 2  ###############
0:	learn: 1.9458235	test: 1.9457935	best: 1.9457935 (0)	total: 68.7ms	remai

In [56]:
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.model_selection import StratifiedKFold

# import gc

# print("Preparing data for Random Forest training...")
# rf_data = train.drop(columns=["id", "Fertilizer Name"])
# rf_labels = train["Fertilizer Name"]
# rf_test_data = test.drop(columns=["id"])
# rf_extra_data = original.drop(columns=["Fertilizer Name"])
# rf_extra_labels = original["Fertilizer Name"]

# # Ensure same column order
# rf_test_data = rf_test_data[rf_data.columns]
# rf_extra_data = rf_extra_data[rf_data.columns]

# # Encode categoricals
# for col in rf_data.columns:
#     if rf_data[col].dtype == 'object':
#         rf_data[col] = rf_data[col].astype('category')
#         rf_test_data[col] = rf_test_data[col].astype('category')
#         rf_extra_data[col] = rf_extra_data[col].astype('category')

# rf_data = pd.get_dummies(rf_data)
# rf_test_data = pd.get_dummies(rf_test_data)
# rf_extra_data = pd.get_dummies(rf_extra_data)

# # Align columns just in case
# rf_test_data = rf_test_data.reindex(columns=rf_data.columns, fill_value=0)
# rf_extra_data = rf_extra_data.reindex(columns=rf_data.columns, fill_value=0)

# # Initialize model
# rf_params = {
#     'n_estimators': 300,
#     'max_depth': 20,
#     'n_jobs': -1,
#     'random_state': 42,
#     'verbose': 0
# }

# rf_folds = 5
# rf_skf = StratifiedKFold(n_splits=rf_folds, shuffle=True, random_state=42)
# rf_predictions = np.zeros((len(rf_test_data), rf_labels.nunique()))

# print("\nStarting 5-Fold training with Random Forest...")
# for fold, (rf_train_idx, rf_valid_idx) in enumerate(rf_skf.split(rf_data, rf_labels)):
#     print('#' * 15, f" RANDOM FOREST FOLD {fold+1} ", '#' * 15)
    
#     rf_x_train = rf_data.iloc[rf_train_idx]
#     rf_x_valid = rf_data.iloc[rf_valid_idx]
#     rf_y_train = rf_labels.iloc[rf_train_idx]
#     rf_y_valid = rf_labels.iloc[rf_valid_idx]

#     # Augment with extra data
#     rf_x_train_aug = pd.concat([rf_x_train, rf_extra_data], axis=0)
#     rf_y_train_aug = pd.concat([rf_y_train, rf_extra_labels], axis=0)

#     rf_model = RandomForestClassifier(**rf_params)
#     rf_model.fit(rf_x_train_aug, rf_y_train_aug)

#     rf_probs = rf_model.predict_proba(rf_test_data)
#     rf_predictions += rf_probs / rf_folds

#     del rf_x_train, rf_x_valid, rf_y_train, rf_y_valid, rf_x_train_aug, rf_y_train_aug, rf_model
#     gc.collect()


In [57]:
# Weighted average of probabilities
ensemble_preds = (0.5 * test_preds) + (0.5 * cb_predictions)




In [58]:
cb_predictions[0]

array([0.15583457, 0.14433669, 0.15175712, 0.11024198, 0.14977104,
       0.16108449, 0.12697412])

In [59]:
top_3_preds_indices = np.argsort(cb_predictions, axis=1)[:, ::-1][:, :3]

decoded_top_3 = target_encoder.inverse_transform(top_3_preds_indices.ravel()).reshape(top_3_preds_indices.shape)

submission = pd.DataFrame({
    'id': test['id'],
    'Fertilizer Name': [' '.join(map(str, row)) for row in decoded_top_3]
})

submission.to_csv('/kaggle/working/submission_cb.csv', index=False)

print("✅ Submission created:")
print(submission.head())


✅ Submission created:
       id         Fertilizer Name
0  750000   DAP 10-26-26 17-17-17
1  750001  17-17-17 Urea 10-26-26
2  750002    20-20 10-26-26 28-28
3  750003       14-35-14 Urea DAP
4  750004      20-20 10-26-26 DAP


In [60]:
print("Classes in encoder:", target_encoder.classes_)
print("Type of classes:", type(target_encoder.classes_[0]))

Classes in encoder: ['10-26-26' '14-35-14' '17-17-17' '20-20' '28-28' 'DAP' 'Urea']
Type of classes: <class 'str'>


In [61]:
print("\nTraining complete. Creating final submission file...")
top_3_preds_indices = np.argsort(ensemble_preds, axis=1)[:, ::-1][:, :3]
top_3_labels = target_encoder.inverse_transform(top_3_preds_indices.ravel()).reshape(top_3_preds_indices.shape)

submission = pd.DataFrame({
    'id': submission['id'],
    'Fertilizer Name': [' '.join(row) for row in top_3_labels]
})

submission.to_csv('/kaggle/working/submission.csv', index=False)
print("✅ Hybrid Feature submission file saved successfully!")
print(submission.head())


Training complete. Creating final submission file...
✅ Hybrid Feature submission file saved successfully!
       id         Fertilizer Name
0  750000   10-26-26 14-35-14 DAP
1  750001  17-17-17 10-26-26 Urea
2  750002          20-20 Urea DAP
3  750003       14-35-14 Urea DAP
4  750004     20-20 Urea 10-26-26
